#Cramer-Rao Bound Calculation

In [ ]:
# Importing modules
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def calculate_stochastic_crb(angles, snr_db, M, d, N):
    """
    Calculates the stochastic CRB RMSE for a specific angle pair and SNR.
    """
    noise_var = 10**(-snr_db/10);

    # Steering matrix
    A = np.column_stack([np.exp(-1j * 2 * np.pi * d * np.arange(M) * np.sin(np.deg2rad(th))) for th in angles])

    # Source and Data Covariance
    R_ss = np.eye(len(angles)) # Signal covariance
    Rxx_true = A @ R_ss @ A.conj().T + noise_var * np.eye(M); # True covariance
    Rxx_inv = np.linalg.inv(Rxx_true);

    # Projection Matrix
    P_A_perp = np.eye(M) - A @ np.linalg.inv(A.conj().T @ A) @ A.conj().T

    # Derivative Matrix
    D_mat = np.zeros((M, len(angles)), dtype=complex);
    m_vec = np.arange(M);
    for i, th in enumerate(angles):
        derivative_factor = -1j * 2 * np.pi * d * m_vec * np.cos(np.deg2rad(th)) * (np.pi / 180.0);
        D_mat[:, i] = derivative_factor * A[:, i];

    # Fisher Information Matrix
    term1 = D_mat.conj().T @ P_A_perp @ D_mat;
    term2 = (R_ss @ A.conj().T @ Rxx_inv @ A @ R_ss).T;
    FIM = (2 * N / noise_var) * np.real(term1 * term2);

    return np.linalg.inv(FIM); # Inverting to get CRB

In [ ]:
# Defining system parameters
M = 16;                # Number of sensors
d = 0.5;               # Sensor spacing (wavelengths)
N = 1000;              # Number of snapshots
D = 2;                 # Number of targets
min_separation = 5.0;  # Minimum angular separation in degrees

ang_min = -60;
ang_max = 60;

# Sweep arrays
snr_db_range = np.arange(-10, 21, 2);                           # SNR from -10 dB to 20 dB
theta1_sweep = np.arange(ang_min, ang_max - min_separation + 0.1, 1.0);  # Sweep the first angle

worst_case_rmse = np.zeros(len(snr_db_range));
worst_case_angles = [];

print(f"Calculating two-target worst-case CRB for {min_separation}° separation")
for i, snr in enumerate(snr_db_range):
    max_rmse = 0;
    worst_pair = (0, 0);

    # Tests all possible adjacent pairs that maintain the minimum separation
    for th1 in theta1_sweep:
        th2 = th1 + min_separation;

        crb = calculate_stochastic_crb([th1, th2], snr, M, d, N);
        rmse = np.sqrt(np.trace(crb)/D);

        current_max_error = np.max(rmse);
        if current_max_error > max_rmse:
            max_rmse = rmse;
            worst_pair = (th1, th2);

    worst_case_rmse[i] = max_rmse;
    worst_case_angles.append(worst_pair);

print("Done.");
print(f"Example worst-case pair at {snr_db_range[0]} dB: {worst_case_angles[0]}");

In [ ]:
# Visualization
plt.figure(figsize=(9, 6));

# Plotting on a logarithmic Y-axis is standard for CRB plots
plt.semilogy(snr_db_range, worst_case_rmse, 'o-', linewidth=2, color='red', label=f'Worst-Case CRB');

plt.title(f'Worst-Case Stochastic CRB vs. SNR\n(ULA, M={M}, N={N}, Min Separation={min_separation}°)');
plt.xlabel('Signal-to-Noise Ratio (dB)');
plt.ylabel('Root Mean Square Error Limit (Degrees)');
plt.grid(True, which="both", ls=":", alpha=0.7);
plt.legend();
plt.tight_layout();
plt.show();

In [ ]:
# Defining system Parameters
M = 16                 # Number of sensors
d = 0.5                # Sensor spacing (wavelengths)
snr_db = 10.0          # Fixed Signal-to-Noise Ratio (dB)
D = 2                  # Number of targets

ang_min = -60;
ang_max = 60;
ang_dist = 5.0        # Minimum angular separation in degrees

# Sweep arrays
N_range = np.linspace(10, 2000, 40).astype(int); # Sweeping N from 10 to 2000 snapshots to clearly see the 1/sqrt(N) curve
theta1_sweep = np.arange(ang_min, ang_max - ang_dist + 0.1, 1.0);

worst_case_rmse = np.zeros(len(N_range));
worst_case_angles = [];

print(f"Calculating two-target worst-case CRB vs Snapshots at SNR = {snr_db} dB");

for i, N_val in enumerate(N_range):
    max_rmse = 0;
    worst_pair = (0, 0);

    # Test all possible adjacent pairs that maintain the minimum separation
    for th1 in theta1_sweep:
        th2 = th1 + min_separation;

        # Calculate the CRB for this specific pair
        crb = calculate_stochastic_crb([th1, th2], snr, M, d, N);
        #rmse = np.sqrt(np.diag(crb));
        rmse = np.sqrt(np.trace(crb)/D);

        current_max_error = rmse; # np.max(rmse)
        if current_max_error > max_rmse:
            max_rmse = rmse;
            worst_pair = (th1, th2);

    worst_case_rmse[i] = max_rmse;
    if i == 0:
        print(f"Worst-case pair found at: {worst_pair}");

print("Done computing.");

In [ ]:
# Visualization
plt.figure(figsize=(9, 6));

# Plotting with a logarithmic Y-axis to see the exponential decay clearly
plt.semilogy(N_range, worst_case_rmse, 'o-', linewidth=2, color='red', label=f'Worst-Case CRB');

plt.title(f'Worst-Case Stochastic CRB vs. Snapshots (N)\n(ULA, M={M}, SNR={snr_db}dB, Min Separation={min_separation}°)');
plt.xlabel('Number of Snapshots (N)');
plt.ylabel('Root Mean Square Error Limit (Degrees)');
plt.grid(True, which="both", ls=":", alpha=0.7);
plt.legend();
plt.tight_layout();
plt.show();